# Proyecto final: Todo modelo es un voto ponderado
**Análisis y Ciencia de Datos — UNIS**  ·  Grupo Bikeshare: Rodolfo Flores, Nohel Estrada, Humberto Najar

Casi todos los modelos del curso predicen como
$$\hat y(x) = \sum_{i=1}^n w_i(x)\, y_i .$$
En este notebook le sacamos los pesos a OLS, ridge, $k$-NN, un árbol y un *random forest*,
comprobamos que reproducen `.predict()`, medimos cuánta evidencia respalda cada predicción
y cuánto cuesta la flexibilidad. La Parte 1 usa California Housing y las Partes 2–4 usan Bikeshare.

Las funciones de pesos están en `pesos.py` (mismo directorio).

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, mean_absolute_error, silhouette_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

import pesos as P

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
FIG = "../figuras"
RS = 0

## Preparación

### 0.1 Partición y estandarización
80/20 con `random_state=0`; el `StandardScaler` se ajusta **solo** con entrenamiento.

In [2]:
# California Housing (Parte 1)
cal = fetch_california_housing(as_frame=True)
Xc_raw_tr, Xc_raw_te, yc_tr, yc_te = train_test_split(cal.data, cal.target, test_size=0.2, random_state=RS)
esc_c = StandardScaler().fit(Xc_raw_tr)
Xc_tr, Xc_te = esc_c.transform(Xc_raw_tr), esc_c.transform(Xc_raw_te)
yc_tr, yc_te = yc_tr.to_numpy(), yc_te.to_numpy()
print("California:", Xc_tr.shape, Xc_te.shape)

California: (16512, 8) (4128, 8)


In [3]:
# Bikeshare (Partes 2 a 5)
bike = pd.read_csv("../data/Bikeshare.csv")
meses = {m: i + 1 for i, m in enumerate(["Jan", "Feb", "March", "April", "May", "June", "July",
                                         "Aug", "Sept", "Oct", "Nov", "Dec"])}
clima = {"clear": 1, "cloudy/misty": 2, "light rain/snow": 3, "heavy rain/snow": 4}
print("meses en el archivo:", bike.mnth.unique())
bike["mnth"] = bike["mnth"].map(meses)
bike["weathersit"] = bike["weathersit"].map(clima)
bike["hr"] = bike["hr"].astype(int)
assert bike[["mnth", "weathersit"]].notna().all().all()

VARS = ["day", "hr", "weekday", "workingday", "holiday", "temp", "hum", "windspeed", "mnth", "weathersit"]
Xb = bike[VARS].astype(float)
yb = np.log1p(bike["bikers"]).to_numpy()
Xb_raw_tr, Xb_raw_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.2, random_state=RS)
esc_b = StandardScaler().fit(Xb_raw_tr)
Xb_tr, Xb_te = esc_b.transform(Xb_raw_tr), esc_b.transform(Xb_raw_te)
bike_tr, bike_te = bike.loc[Xb_raw_tr.index], bike.loc[Xb_raw_te.index]
n_b = len(Xb_tr)
print("Bikeshare:", Xb_tr.shape, Xb_te.shape)

meses en el archivo: <StringArray>
['Jan', 'Feb', 'March', 'April', 'May', 'June', 'July', 'Aug', 'Sept', 'Oct', 'Nov', 'Dec']
Length: 12, dtype: str
Bikeshare: (6916, 10) (1729, 10)


### 0.2 Taiwan
No aplica a nuestro grupo (usamos Bikeshare), pero la idea es útil: si $w_i(x)\ge 0$ y $\sum_i w_i(x)=1$,
el voto ponderado de etiquetas 0/1 es un promedio convexo y queda en $[0,1]$. $k$-NN, el árbol y el bosque
cumplen esa condición; OLS y ridge suman 1 pero pueden tener pesos negativos.

### 0.3 ¿Por qué `casual` y `registered` no pueden ser variables?

In [4]:
print("¿casual + registered == bikers en todas las filas?",
      bool((bike.casual + bike.registered == bike.bikers).all()))

¿casual + registered == bikers en todas las filas? True


Porque son la respuesta partida en dos: `bikers = casual + registered` en el 100 % de las filas.
Usarlas sería *fuga de información* (target leakage): el modelo "predeciría" con la respuesta misma
y además, a la hora de usarlo (planear la reubicación de bicicletas por la mañana), esas cantidades
todavía no se conocen.

### 0.4 Desviación estándar antes de estandarizar
La guardamos en `desv_cruda`; la usamos en la Parte 4.

In [5]:
desv_cruda = Xb_raw_tr.std(ddof=0)
pd.DataFrame({"desv. estándar (entrenamiento, crudo)": desv_cruda,
              "media": Xb_raw_tr.mean()}).sort_values("desv. estándar (entrenamiento, crudo)", ascending=False)

,"desv. estándar (entrenamiento, crudo)",media
day,104.9296,184.6666
hr,6.8997,11.5759
mnth,3.4313,6.5805
weekday,2.0025,2.9935
weathersit,0.6549,1.4377
workingday,0.4649,0.6841
temp,0.1978,0.4894
hum,0.1968,0.6434
holiday,0.1668,0.0286
windspeed,0.1227,0.1904
